In [13]:
import os

PROJECT_ROOT = "/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation"
os.chdir(PROJECT_ROOT)

print("CWD:", os.getcwd())

CWD: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation


First we will get the rsIDs from cohort1 and cohort2, but because the annotated BP's do are not always up to date with the latest Ensemble GRCh38 we will scrape the necessary information from ensemble.

In [ ]:
#!/usr/bin/env python3
import pandas as pd
import requests
import time

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

rsids = sorted(set(pd.read_csv("chronic_data/latent_var_cohort1.csv")["SNP"].astype(str)
                   .str.replace("^GSA-", "", regex=True)
                   .tolist()
                 + pd.read_csv("chronic_data/latent_var_cohort2.csv")["SNP"].astype(str)
                   .str.replace("^GSA-", "", regex=True)
                   .tolist()))

# keep only things that look like rsIDs
rsids = [r for r in rsids if r.startswith("rs")]

out_rows = []
url = "https://rest.ensembl.org/vep/human/id"
headers = {"Content-Type": "application/json", "Accept": "application/json"}

for batch in chunked(rsids, 200):  # 200 is safe
    payload = {"ids": batch}
    r = requests.post(url, headers=headers, json=payload)
    r.raise_for_status()
    data = r.json()

    for item in data:
        rsid = item.get("id")
        # Most entries have "seq_region_name" + "start"/"end"
        chrom = item.get("seq_region_name")
        start = item.get("start")
        end = item.get("end")

        # Genes: prefer "transcript_consequences" gene_symbol/gene_id if present
        genes = set()
        for tc in item.get("transcript_consequences", []) or []:
            if "gene_symbol" in tc and tc["gene_symbol"]:
                genes.add(tc["gene_symbol"])
            elif "gene_id" in tc and tc["gene_id"]:
                genes.add(tc["gene_id"])

        # fallback: nearby gene info can appear in "colocated_variants" etc., but transcripts are usually enough
        out_rows.append({
            "rsid": rsid,
            "chr": chrom,
            "start_GRCh38": start,
            "end_GRCh38": end,
            "genes": ";".join(sorted(genes)) if genes else ""
        })

    time.sleep(0.1)  # be polite to API

df = pd.DataFrame(out_rows).drop_duplicates(subset=["rsid"])
df.to_csv("chronic_data/ensembl/rsid_to_genes_GRCh38.tsv", sep="\t", index=False)
print("Wrote chronic_data/ensembl/rsid_to_genes_GRCh38.tsv")

The file created above has sometimes multiple genes on one rsID. To take care of this we expand the file.

In [ ]:
import pandas as pd

df = pd.read_csv("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/chronic_data/ensembl/rsid_to_genes_GRCh38.tsv", sep="\t")

df["genes"] = df["genes"].fillna("")
df = df.assign(gene=df["genes"].str.split(";")).explode("gene")
df = df[df["gene"] != ""]

df.to_csv("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/chronic_data/ensembl/rsid_gene_expanded.tsv", sep="\t", index=False)
print("Wrote chronic_data/ensembl/rsid_genes_expanded.tsv")

Now we will join the GWAS statistics from latent_var_cohort1 and latent_var_cohort2


In [18]:
gwas = pd.read_csv("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/chronic_data/latent_var_cohort1.csv")
map_df = pd.read_csv("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/chronic_data/ensembl/rsid_gene_expanded.tsv", sep="\t")

merged = gwas.merge(
    map_df,
    left_on="SNP",
    right_on="rsid",
    how="left"
)

merged.to_csv("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/chronic_data/ensembl/gwas_snps_with_genes.tsv", sep="\t", index=False)
print("Wrote chronic_data/ensembl/gwas_snps_with_genes.tsv")

Wrote chronic_data/ensembl/gwas_snps_with_genes.tsv


Before we can overlap the data with the burn data, we need to collapse GWAS SNPs to gene-level signals (like we have in the burn data).
Currently there are some genes with multiple SNPs in the data. If we keep these, then they will dominate.


In order to decide which GWAS signals to use, we look at the BETA, STAT and P statistics in the chronic data. We then can do the either of the following:
- abs(BETA) for effect size
- -log10(P) for significance

We then take the maximum across SNPs mapping to that gene